# P1a: task–backbone applicability confidence audit

This CPU-only notebook reuses the unchanged P0c calibration score table. It computes clustered bootstrap confidence bounds for each `(backbone, task)` group and applies the newly frozen applicability rule. It performs **no model inference** and never instantiates sealed evaluation origins. The output is `screening_only`, not paper evidence.

Use a **CPU runtime**. Expected runtime is roughly 2–10 minutes.

In [ ]:
import importlib
import subprocess
import sys
from pathlib import Path

REPO = Path('/content/covariate-safe-tsfm')
if not REPO.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/FlyMe2star/covariate-safe-tsfm.git', str(REPO)],
        check=True,
    )
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)],
    check=True,
)
SOURCE_ROOT = str(REPO / 'src')
if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)
importlib.invalidate_caches()
importlib.import_module('covsafe')
print('Repository ready:', REPO)
print('Git commit:', subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True
).strip())

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
PRIVATE_ROOT = Path(
    '/content/drive/MyDrive/covariate-safe-tsfm/private_manifests'
)
P0C_ROOT = PRIVATE_ROOT / 'p0c'
P1A_ROOT = PRIVATE_ROOT / 'p1a'
assert (P0C_ROOT / 'reports/p0c_predictability_screen.json').exists(), (
    'P0c report is missing from Google Drive.'
)
assert (P0C_ROOT / 'scores/p0c_prequential_scores.csv').exists(), (
    'P0c score table is missing from Google Drive.'
)
P1A_ROOT.mkdir(parents=True, exist_ok=True)
print('P0c input root:', P0C_ROOT)
print('P1a durable root:', P1A_ROOT)

In [ ]:
import json

from covsafe.p1a import EXPECTED_P1A_CONFIG_HASH, run_p1a

print('Frozen P1a config hash:', EXPECTED_P1A_CONFIG_HASH)
report = run_p1a(REPO, P0C_ROOT, P1A_ROOT)
print(json.dumps(report, indent=2, ensure_ascii=False, default=str))

In [ ]:
summary = {
    backbone: {
        task: {
            'point_auroc': result['bootstrap']['point_auroc'],
            'lower_bound': result['bootstrap']['one_sided_lower_bound'],
            'eligible': result['eligible'],
        }
        for task, result in tasks.items()
    }
    for backbone, tasks in report['groups'].items()
}
print(json.dumps({
    'groups': summary,
    'continuation_gate': report['continuation_gate'],
}, indent=2, ensure_ascii=False))

## Return artifact

Send the final compact JSON containing `groups` and `continuation_gate`. Do not open or run any sealed-evaluation notebook even if the gate passes; a separate P1b router contract must be frozen first.